# Tarea 12

> Uriel Aguilar Mauricio

## Lectura, Análisis Crítico y Discusión de un Artículo Clásico de Predicción de Defectos de Software

**Artículo analizado:**  
Menzies, T., Greenwald, J., & Frank, A. (2007). *Data mining static code attributes to learn defect predictors*. IEEE Transactions on Software Engineering, 33(1), 2-13.

## Parte 1. Análisis del artículo

| **Elemento** | **Descripción** |
|---|---|
| Problema de investigación | Existía un debate abierto sobre si los atributos estáticos del código (líneas de código, McCabe, Halstead) sirven para predecir módulos defectuosos. Los estudios previos llegaban a conclusiones contradictorias (unos a favor, otros, como Fenton y Pfleeger o Shepperd e Ince, en contra) y además usaban datos distintos, por lo que no eran comparables. El problema es doble: demostrar si estos predictores son realmente útiles y proponer un experimento base (baseline) repetible con datos públicos. |
| Hipótesis o pregunta principal | Si los críticos tuvieran razón, los atributos de McCabe/Halstead no serían mejores que un simple umbral sobre líneas de código y los predictores aprendidos serían muy pobres. Los autores sostienen lo contrario: la pregunta central es si un minero de datos bien elegido logra predictores útiles, y si *cómo* se usan los atributos (el método de aprendizaje) importa más que *cuáles* atributos se usan. |
| Datos utilizados | Ocho conjuntos de datos del repositorio NASA MDP (cm1, kc3, kc4, mw1, pc1, pc2, pc3, pc4), provenientes de cuatro sistemas (instrumento de nave espacial, gestión de almacenamiento de datos en tierra, base de datos y software de vuelo de un satélite). Cada registro es un módulo (la unidad funcional más pequeña). Se eliminó el identificador, se descartó la densidad de error y la cantidad de errores se convirtió en una clase binaria (defective? = error_count mayor o igual a 1). Quedaron 38 atributos más la clase. El desbalance es fuerte: de 49% de módulos defectuosos en kc4 hasta solo 0.4% en pc2. |
| Métricas de software empleadas | Atributos estáticos agrupados en: (a) conteos de líneas (loc total, código, comentarios, blancos, ejecutables); (b) McCabe: complejidad ciclomática v(g), de diseño iv(g) y esencial ev(g); (c) Halstead: conteos crudos de operadores y operandos (únicos y totales) y ocho derivados (longitud, volumen, nivel, dificultad, contenido, esfuerzo, error estimado y tiempo de programación); y (d) atributos misceláneos poco documentados (branch_count, call_pairs, condition_count, edge_count, node_count, etc.). Para evaluar los predictores se usaron pd (probabilidad de detección), pf (probabilidad de falsa alarma), notPf y balance (bal), no la exactitud. |
| Algoritmos o técnicas utilizadas | Tres clasificadores de WEKA: OneR (regla de umbral sobre un único atributo), J48 (árbol de decisión, implementación de C4.5) y Naive Bayes. Dos filtros de preprocesamiento numérico: ninguno o logNums (reemplazar cada valor por su logaritmo natural, con un piso de 0.000001). Selección de atributos mediante InfoGain (ganancia de información), de forma iterativa y exhaustiva. Evaluación con validación cruzada 10x10 (diez repeticiones con orden aleatorio y 10 particiones), es decir, un esquema holdout. Resultados resumidos con gráficas de cuartiles de diferencias de desempeño. |
| Resultados principales | Naive Bayes con logNums fue el mejor método en pd y en balance (mediana de balance de 22.1 y mediana de pd de 52.4 en las diferencias), a costa de una peor notPf, lo cual es esperable porque aumentar pd implica más falsas alarmas. J48 superó a OneR, y OneR tuvo medianas negativas. Con solo 2 o 3 atributos seleccionados por InfoGain se logró el mismo balance que con los 38, pero con 1 solo atributo el desempeño empeoró. Con atributos seleccionados, el promedio sobre los ocho conjuntos fue pd = 71% y pf = 25%. En pc2, con apenas 0.4% de módulos defectuosos, se obtuvo pd = 72% y pf = 14%. Los mejores atributos cambian de un conjunto a otro porque la ganancia de información de los atributos principales es muy similar. |
| Conclusiones de los autores | (1) Los predictores basados en atributos estáticos sí son útiles y superan a métodos industriales como las revisiones manuales (alrededor de 60% de detección según el panel de IEEE Metrics 2002; menos aún según Raffo). (2) Debatir 'LOC contra McCabe' está mal enfocado: importa más el método de aprendizaje que el subconjunto de atributos. (3) El espacio de 'mejores atributos' es frágil, lo que explica la contradicción de la literatura previa. (4) No se recomiendan umbrales simples ni árboles complejos; conviene un método que combine muchos indicadores parciales, como Naive Bayes con transformación logarítmica. (5) Los predictores deben usarse como indicadores probabilísticos, no como oráculos, para priorizar recursos limitados de verificación y validación. (6) Ya no basta evaluar con un solo conjunto de datos y un solo aprendiz. |

## Parte 2. Evaluación crítica de la metodología



### 1. ¿Es adecuada la selección de datos?

En términos generales la considero adecuada para su época, pero con reservas. A favor: los datos provienen de un repositorio público (NASA MDP), abarcan ocho subsistemas de cuatro sistemas distintos, desarrollados en diferentes ubicaciones y sin intercambio de personal ni código entre subsistemas, lo que reduce la dependencia entre muestras. Además, hay variedad de tamaños (de 126 a 5,590 módulos) y de proporciones de defectos (0.4% a 49%), lo que permite observar el comportamiento ante desbalance de clases. Sobre todo, al ser públicos, permiten que otros repliquen el trabajo, que era el objetivo declarado de definir un baseline.
Todos los datos son de NASA y de contratistas aeroespaciales, un nicho con procesos de calidad particulares; los autores defienden su generalización, pero es un argumento por analogía y no una demostración. Además, los atributos 'misceláneos' están mal documentados y no se pueden reproducir fuera del repositorio, y la etiqueta de defecto depende de los registros de defectos de cada proyecto, que pueden estar incompletos o ser ruidosos. Años después se documentó que algunos conjuntos de NASA tenían problemas de calidad (duplicados, inconsistencias), un riesgo que el artículo no examina.

### 2. Principales fortalezas metodológicas

- **Evaluación con holdout (10x10 cross-validation)** en lugar de auto-pruebas, que sobrestiman el desempeño. Esto da estimaciones más honestas de cómo se comportaría el predictor con módulos nuevos.
- **Rechazo de la exactitud (accuracy)** como métrica por el desbalance de clases. Un clasificador que siempre predice 'sin defecto' logra 90% de exactitud con 10% de defectuosos y no sirve de nada. El uso de pd, pf y balance (distancia al punto ideal pd = 1, pf = 0 en el espacio ROC) es técnicamente más apropiado.
- **Repetición con aleatorización del orden** para controlar efectos de orden, lo que aporta robustez.
- **Comparación de varios aprendices y filtros** (OneR, J48, Naive Bayes; con y sin logaritmos) frente a un solo método, lo que sustenta la afirmación de que el método importa más que los atributos.
- **Análisis no paramétrico con gráficas de cuartiles**, razonable porque las distribuciones de las diferencias de desempeño son claramente no gaussianas (por ejemplo, más de la mitad de los resultados de pd de OneR toman el mismo valor). Aun así reportan pruebas t por completitud.
- **Transparencia:** publican amenazas a la validez, el procedimiento y los datos, y buscan que el artículo funcione como línea base.

### 3. Limitaciones del diseño experimental

- **Selección de atributos dentro del experimento.** En la Fig. 7 el ranking por InfoGain se calcula sobre los datos antes de dividir en entrenamiento y prueba. Si el ranking usa información de los datos de prueba, hay un riesgo de fuga de información (data leakage) que podría inflar los resultados. Es el punto técnicamente más delicado del diseño.
- **Pocos aprendices.** Solo tres clasificadores; los propios autores excluyen redes neuronales y algoritmos genéticos por costo computacional. La conclusión de que Naive Bayes es el mejor se limita a ese conjunto.
- **Ajuste de hiperparámetros ausente.** J48 y OneR se usaron con configuraciones estándar; Naive Bayes se beneficia del logaritmo, pero no se explora si J48 mejoraría con poda o ajuste.
- **Costo de las falsas alarmas.** El método ganador mejora pd pero con pf de 25%. El artículo no cuantifica el costo real de inspeccionar módulos de más frente a perder defectos, algo clave en la práctica.
- **Granularidad y alcance.** La unidad es el módulo; no se distinguen defectos graves de leves y se hace predicción dentro del mismo proyecto (con partición aleatoria), no entre proyectos ni entre versiones en el tiempo, que es el escenario más realista.
- **Pruebas estadísticas.** Se apoyan en efectos 'destacados' observados en gráficas de cuartiles más que en pruebas formales de significancia con corrección por comparaciones múltiples.
- **Supuesto de independencia de Naive Bayes** con atributos muy correlacionados (loc, v(g), conteos de Halstead); los autores no discuten el efecto de esa correlación.

### 4. Amenazas a la validez

- **Validez externa:** sesgo de muestreo hacia software de NASA/aeroespacial, en lenguajes C y Java de la época; los resultados podrían no trasladarse a software web, móvil o de código abierto moderno.
- **Validez de constructo:** 'defectuoso' se define como al menos un error registrado, y los defectos no reportados quedan como 'sin defecto'. Además, los atributos son un reflejo limitado del código (por ejemplo, el ejemplo de Fenton y Pfleeger donde la misma funcionalidad produce métricas distintas).
- **Validez interna:** posible fuga de información en la selección de atributos, efectos de orden (mitigados con repeticiones), y datos con calidad no verificada.
- **Validez de conclusión:** uso de estadística paramétrica sobre distribuciones sesgadas y comparaciones múltiples, lo cual los autores reconocen y mitigan parcialmente con cuartiles.
- **Reproducibilidad:** aunque los datos son públicos, el estudio tardó semanas en depurarse y usa una versión de WEKA de la época; pequeñas diferencias de implementación podrían cambiar los números.
- **Sesgo de mejor resultado:** el 71%/25% reportado proviene de elegir el mejor subconjunto de 2 o 3 atributos por conjunto de datos, lo que puede ser algo optimista frente al desempeño con una selección fija.

## Parte 3. Comparación con los Temas Estudiados en el Curso

### ¿Qué métricas aparecen tanto en el artículo como en las actividades realizadas con KC1?

El conjunto KC1 del repositorio NASA MDP (usado en las actividades del curso) es de la misma familia que los datos del artículo (kc3 y kc4 provienen del mismo sistema de gestión de almacenamiento de datos en tierra). Por ello comparte los mismos grupos de métricas: **líneas de código** (loc, lOCode, lOComment, lOBlank, lOCodeAndComment), **McCabe** (v(g), ev(g), iv(g)), **Halstead** (n, v, l, d, i, e, b, t, y los conteos de operadores y operandos únicos y totales) y atributos de estructura como branchCount. La variable objetivo también es binaria (con o sin defectos). La diferencia principal es que el artículo trabaja con 38 atributos tras su preprocesamiento (incluye misceláneos como call_pairs, edge_count y node_count), mientras que KC1 suele distribuirse con alrededor de 21 atributos. Esto muestra que las métricas clásicas (LOC, McCabe, Halstead) siguen siendo la base de los datasets de defectos que se usan en el curso.

### ¿Qué relación encuentra entre el artículo y los métodos de selección de características estudiados?

El artículo dedica una parte sustancial a la selección de atributos (*attribute subset selection*). Usa **InfoGain**, un método de tipo filtro basado en entropía, que ordena los atributos por cuánta incertidumbre de la clase reducen: InfoGain(A) = H(C) - H(C|A). Aplica una selección iterativa (probar los 1, 2, ..., N mejores) y una exhaustiva (todos los subconjuntos de los j mejores). También menciona CFS, Relief y CBS, sin ventaja apreciable sobre InfoGain. El hallazgo más interesante para el curso es que 2 o 3 atributos igualan el desempeño de los 38, es decir, hay mucha redundancia (alta correlación entre LOC, v(g) y las métricas de Halstead), pero un solo atributo no basta. Además, los atributos elegidos varían entre conjuntos de datos y entre submuestras (Fig. 14), lo que muestra la inestabilidad de la selección: pequeños cambios en la muestra de entrenamiento cambian el 'mejor' atributo. Esto conecta con lo visto en clase sobre reducción de dimensionalidad, redundancia y el riesgo de sobreajustar el subconjunto seleccionado.

### ¿Cómo se relaciona el trabajo con Naive Bayes y otros clasificadores supervisados?

El artículo es un caso de estudio de clasificación supervisada binaria. **Naive Bayes** aplica el teorema de Bayes con el supuesto de independencia condicional entre atributos, y para atributos numéricos modela cada uno con una distribución normal. Como las métricas de código siguen distribuciones muy sesgadas (exponenciales: muchos valores pequeños y pocos muy grandes, Fig. 6), la transformación logarítmica las acerca a una forma más gaussiana y mejora justamente la hipótesis que Naive Bayes asume. Esa es la explicación técnica que dan los autores de por qué logNums.nb gana. Los otros dos clasificadores muestran dos extremos: **OneR** (una regla sobre un solo atributo, un árbol de profundidad 1) rinde peor y **J48/C4.5** (árbol con criterio de ganancia de información y búsqueda voraz) rinde mejor que OneR pero peor que Naive Bayes. Además, la evaluación con validación cruzada, la matriz de confusión (celdas A, B, C, D), la curva ROC y las métricas recall (pd), tasa de falsos positivos (pf) y balance son los mismos conceptos de evaluación de clasificadores vistos en el curso.

### ¿Qué conceptos del curso le ayudaron a comprender mejor el artículo?

- **Matriz de confusión y métricas derivadas** (recall, tasa de falsos positivos, precisión, exactitud): permiten entender por qué pd y pf son centrales y por qué la exactitud engaña con clases desbalanceadas.
- **Desbalance de clases:** explica la decisión de no usar accuracy y el mérito del resultado en pc2 (0.4% de defectuosos).
- **Validación cruzada y overfitting:** permiten entender la crítica a las auto-pruebas y el esquema 10x10.
- **Entropía y ganancia de información:** base de InfoGain y de los árboles de decisión.
- **Preprocesamiento (transformaciones y discretización):** el efecto del logaritmo sobre la distribución de las métricas.
- **Compromiso sesgo-varianza y simplicidad del modelo:** OneR (alto sesgo) frente a J48 (más flexible) frente a Naive Bayes (pocos parámetros, estable).

## Parte 4. Discusión y reflexión personal

### 1. ¿Considera que las métricas de software son suficientes para predecir defectos?

Considero que **no son suficientes por sí solas, pero sí son útiles como indicadores**. Las métricas estáticas capturan tamaño y complejidad estructural, y la evidencia del artículo (pd de 71% con pf de 25%) muestra que contienen señal. Sin embargo, los propios autores aceptan que son 'probabilísticas' y que no capturan la semántica del código: dos fragmentos con la misma funcionalidad pueden tener métricas muy distintas. Un defecto depende también de factores humanos y de proceso que no aparecen en el código: la experiencia del desarrollador, la historia de cambios (code churn), la cantidad de autores, la antigüedad del módulo, el acoplamiento entre módulos, la calidad de los requisitos y la cobertura de pruebas. Estudios posteriores (por ejemplo, métricas de proceso y de cambios) han mostrado que combinarlas con las métricas estáticas mejora la predicción. Por tanto, las métricas estáticas son un buen punto de partida, barato y automático, pero deben complementarse.

### 2. ¿Los resultados del artículo siguen siendo relevantes casi veinte años después de su publicación? Justifique.

En parte sí y en parte no. **Sigue vigente** la idea central de que el método de aprendizaje y la evaluación cuidadosa importan más que debatir qué métrica es mejor, la advertencia contra métricas como accuracy en datos desbalanceados, la recomendación de validar con datos no vistos, y el uso de los predictores como herramienta de priorización de esfuerzo de pruebas y no como oráculo. También sigue siendo útil el hallazgo de que pocas métricas bastan y de que la selección de atributos es inestable. **Ha perdido vigencia** la afirmación de que Naive Bayes es la mejor opción: hoy los métodos de ensamble (Random Forest, gradient boosting) y el aprendizaje profundo suelen superarlo; también se sabe que la validación aleatoria dentro de un proyecto es optimista frente a la predicción entre versiones o entre proyectos (cross-project), que el desbalance debe tratarse activamente (SMOTE, costos), y que los conjuntos de NASA tenían problemas de calidad de datos. Además, el contexto cambió: hoy se desarrolla con integración continua, repositorios de control de versiones y código abierto, de modo que existen predictores a nivel de cambio (just-in-time) con información que en 2007 no se usaba. Aun así, el artículo sigue siendo un clásico metodológico.

### 3. Si usted repitiera el estudio actualmente, ¿qué cambios introduciría?

- **Corregir el posible data leakage** haciendo el ranking y la selección de atributos solo con el conjunto de entrenamiento de cada partición (por ejemplo, dentro de un pipeline con validación cruzada anidada).
- **Mejorar la calidad de los datos:** usar versiones depuradas de los conjuntos NASA (eliminar duplicados e inconsistencias) y añadir conjuntos modernos de proyectos de código abierto (por ejemplo, PROMISE, ECLIPSE, o datasets con historial de commits).
- **Validación más realista:** separación temporal (entrenar con versiones pasadas y probar con versiones futuras) y experimentos entre proyectos (cross-project), no solo particiones aleatorias.
- **Tratar el desbalance** con remuestreo (SMOTE, submuestreo) o aprendizaje sensible al costo, y reportar más métricas: AUC-ROC, F1, MCC, precisión y recall en la clase positiva, junto con balance.
- **Ajuste de hiperparámetros** para todos los modelos (búsqueda aleatoria o bayesiana), de modo que la comparación sea justa.
- **Pruebas estadísticas formales** (Wilcoxon, Friedman con Nemenyi o Scott-Knott) con corrección por comparaciones múltiples y tamaños de efecto.
- **Análisis de costo-beneficio** y de esfuerzo (por ejemplo, cuánto código hay que revisar para encontrar un porcentaje dado de defectos).
- **Incluir métricas de proceso y de cambio** (churn, autores, historia de defectos) junto con las estáticas.

### 4. ¿Qué nuevos algoritmos o técnicas modernas agregaría al análisis?

- **Modelos de ensamble:** Random Forest, Gradient Boosting, XGBoost, LightGBM y CatBoost, que manejan bien relaciones no lineales y atributos correlacionados.
- **Regresión logística y SVM** como líneas base adicionales, y **redes neuronales** (MLP) como comparación.
- **Aprendizaje profundo sobre el código fuente:** representaciones del árbol sintáctico abstracto (AST) con redes convolucionales o recurrentes (LSTM), y modelos basados en Transformers para código como CodeBERT, que aprenden características semánticas sin depender de métricas manuales.
- **Técnicas para desbalance:** SMOTE y variantes, ensambles balanceados (BalancedRandomForest) y funciones de pérdida sensibles al costo.
- **Selección de características más moderna:** métodos wrapper o embedded (LASSO, importancia por permutación), reducción con PCA, y estabilidad de la selección.
- **Explicabilidad:** SHAP y LIME, para entender qué métricas influyen en cada predicción y dar confianza a los ingenieros.
- **Transferencia y predicción entre proyectos:** transfer learning y adaptación de dominio; también aprendizaje 'just-in-time' a nivel de commit.

### 5. ¿Cuál considera que fue la contribución más importante del artículo para el área de predicción de defectos?

A mi parecer, la contribución más importante fue **cambiar la pregunta de investigación**: pasar de '¿qué métrica es mejor?' a '¿qué método de aprendizaje y qué protocolo de evaluación son adecuados?'. Con ello el artículo (i) estableció un **baseline experimental repetible** con datos públicos que permitió comparar trabajos futuros, (ii) mostró de forma convincente que **la accuracy es inapropiada** con clases desbalanceadas y popularizó pd, pf y balance, (iii) explicó la contradicción de la literatura previa mediante la **fragilidad del conjunto de 'mejores' atributos**, y (iv) demostró que, contra el pesimismo de la época, los predictores de defectos basados en métricas estáticas son **prácticamente útiles** para priorizar la inspección y las pruebas. Esa postura metodológica, que exige varios aprendices, varios conjuntos de datos y validación holdout, dio forma a buena parte de la investigación posterior sobre predicción de defectos.

## Referencias bibliográficas

- Menzies, T., Greenwald, J., & Frank, A. (2007). Data mining static code attributes to learn defect predictors. *IEEE Transactions on Software Engineering, 33*(1), 2-13.